# 05 · The real test, and the weekly plan

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EngMtaleb/chemai/blob/main/projects/p03_ro_fouling/notebooks/05_real_test_and_plan.ipynb)

Everything was chosen on simulated development plants and checked once on simulated test plants.
Then, before the real deposit was touched, this was written down (`results/step6_declared_before_run.md`,
verbatim):

> **Declared before the external test was run**
>
> Written after steps 4 (development plants) and 5 (test plants), before step 6 touched the
> forecasts on the real deposit.
>
> - Frozen: linear law, 42-day trailing window, 3 flushing days skipped, interval inflation 2.0
>   (P10-P90), flag beyond 2x the observed window, plan a CIP when the P50 date is within 28 days.
>   Review levels scored: 1.15 and 1.20.
> - Primary analysis: the linear law. Secondary, reported but not adopted: the gap law
>   (DP_norm^(-1/2.2)), which was 1-2 days better on simulated plants - a gain that may be built in,
>   because the simulator itself uses the gap law.
> - Expectation: the deposit's DP_norm was normalised by its authors with a vendor-style flow
>   exponent (1.5, per operating mode). On simulated plants a vendor exponent cost about 4 days of
>   median error and let a simple margin rule beat the forecast. If the real plant's exponent is
>   nearer the physics (1.7-1.8), the same may happen here. A forecast that does not beat the margin
>   rule on the deposit will be reported as such.
> - Unit of every score: the cleaning cycle. Days inside a cycle are not independent; the cycle
>   bootstrap gives the intervals.

This notebook runs that test — once — and then builds the weekly plan.

**Data:** Su, Y.H., Meng, F.C. and Chang, P.Y. (2026) *Auditable clean-in-place decision support from routine SWRO SCADA.* Membranes 16(8), 257, [doi:10.3390/membranes16080257](https://doi.org/10.3390/membranes16080257). Deposit: [doi:10.5281/zenodo.21055948](https://doi.org/10.5281/zenodo.21055948), CC-BY-4.0. The plant and its owner are not named by the authors and are not named here.

Script version: `step6_external_test.py` · record: `VALIDATION.md`, sections 11–12.

In [1]:
# Setup. In Google Colab this installs the package from GitHub (about a minute).
# On your own machine, run from a clone after:  pip install -e ".[viz]"
import os, subprocess, sys
REF = "main"        # before the Project 3 pull request is merged, use "p03-ro-fouling"
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    f"chemai[viz] @ git+https://github.com/EngMtaleb/chemai.git@{REF}"], check=True)
    os.environ["CHEMAI_DATA_DIR"] = "/content/data"     # where the deposit is downloaded

import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25,
                     "axes.spines.top": False, "axes.spines.right": False})
pd.set_option("display.width", 140)
COL = {"A": "#2a6fb0", "B": "#128c84", "C": "#a8782f"}

In [2]:
from chemai.config import RODepositConfig, ROConfig
from chemai.data.ro_deposit import download_deposit, load_daily, load_ledger, split_cycles
from chemai.evaluation.fouling_eval import real_decisions, score_real

dcfg, cfg = RODepositConfig(), ROConfig()
folder = download_deposit(cfg=dcfg)
daily, ledger = load_daily(folder / dcfg.daily_file), load_ledger(folder / dcfg.ledger_file)
cycles = split_cycles(daily, dcfg)

# weekly decisions in every real cycle; truth = first day the 7-day median reaches the level,
# censored ("not before day x") when a cleaning came first
cr, lv = score_real(cycles, (1.15, 1.20), "linear", cfg)
print(len(cr), "decisions;", (cr["status"] == "insufficient").mean().round(2), "had too few recorded days")
cr["status"].value_counts()

596 decisions; 0.41 had too few recorded days


status
insufficient    242
not_in_sight    161
at_or_above     109
crossing         77
falling           7
Name: count, dtype: int64

## 1. Crossing dates

In [3]:
g = cr[(cr["truth"] > 0) & (cr["status"] != "insufficient")]
unc = g[~g["truth_censored"] & (g["truth"] <= 180)]                 # crossings that were observed
dated = unc[unc["status"] == "crossing"]
err = (dated["p50"] - dated["truth"])
print(f"crossings observed: {len(unc)}; dated by the forecast: {len(dated)}")
print(f"median absolute error: {err.abs().median():.1f} days; dated late (P50 after the truth): {(err > 0).mean():.0%}")
inside, flagged = dated[dated["in_envelope"]], dated[~dated["in_envelope"]]
print(f"median error, unflagged: {(inside['p50'] - inside['truth']).abs().median():.1f} days (n={len(inside)})"
      f" | flagged: {(flagged['p50'] - flagged['truth']).abs().median():.1f} days (n={len(flagged)})")
print("\nwhat the forecast said before the crossings it did not date:")
print(unc[unc["status"] != "crossing"]["status"].value_counts().to_string())
nis = unc[unc["status"] == "not_in_sight"]
print(f"'not in sight': DP_norm then (median) {nis['level_now'].median():.3f}; crossed a median {nis['truth'].median():.0f} days later")

crossings observed: 151; dated by the forecast: 54
median absolute error: 28.4 days; dated late (P50 after the truth): 69%
median error, unflagged: 12.5 days (n=30) | flagged: 59.2 days (n=24)

what the forecast said before the crossings it did not date:
status
not_in_sight    85
at_or_above     10
falling          2
'not in sight': DP_norm then (median) 1.007; crossed a median 75 days later


Most real crossings were never dated: the pressure drop sat on its clean baseline, the line had
no slope to extend, and the rise came later. The dates it did give were mostly late — and the late
ones are the flagged ones. The flag did its job.

## 2. Level forecasts against "it stays where it is"

In [4]:
lv["err_forecast"] = (lv["predicted"] - lv["measured"]).abs()
lv["err_persistence"] = (lv["persistence"] - lv["measured"]).abs()
lv.groupby("horizon").agg(decisions=("cycle", "size"), cycles=("cycle", "nunique"),
                          forecast_mae=("err_forecast", "mean"), persistence_mae=("err_persistence", "mean")).round(4)

,decisions,cycles,forecast_mae,persistence_mae
horizon,,,,
14,113,15,0.0338,0.0342
28,85,7,0.0525,0.0558


No better than assuming tomorrow equals today.

## 3. The decision: will it cross within 28 days?

In [5]:
d = real_decisions(cr)                       # decisions whose answer is known
y = d["y"].astype(bool).to_numpy()
fc = ((d["status"] == "crossing") & (d["p50"].fillna(np.inf) <= 28)).to_numpy()

def row(name, f):
    tp, fp, fn = (f & y).sum(), (f & ~y).sum(), (~f & y).sum()
    return dict(rule=name, caught=tp, false_alarms=fp, recall=round(tp / (tp + fn), 2), precision=round(tp / max(tp + fp, 1), 2))

rows = [row("forecast: P50 within 28 days (frozen)", fc)]
for m in (0.02, 0.03, 0.05, 0.08):
    rows.append(row(f"margin {m}", (d["obs_level_now"] >= d["level"] - m).to_numpy()))
rows.append(row("both - adopted AFTER this test", fc | (d["obs_level_now"] >= d["level"] - 0.03).to_numpy()))
print(f"{len(d)} decisions with a known answer, {y.sum()} crossed within 28 days")
pd.DataFrame(rows)

213 decisions with a known answer, 61 crossed within 28 days


,rule,caught,false_alarms,recall,precision
0,forecast: P50 within 28 days (frozen),13,5,0.21,0.72
1,margin 0.02,21,1,0.34,0.95
2,margin 0.03,26,2,0.43,0.93
3,margin 0.05,41,7,0.67,0.85
4,margin 0.08,47,19,0.77,0.71
5,both - adopted AFTER this test,31,6,0.51,0.84


**The forecast lost to the margin rule at every operating point** — as the declaration said it
might, for the reason it named: the deposit's pressure drop carries a vendor normalisation. A second
reason was found afterwards: the real cycles are short and quiet, so a train near the level crosses
on its day-to-day swings before any trend matters.

The weekly plan therefore uses **both** triggers. That change was made after seeing this result,
so the last row is optimistic. On simulated plants — the development and test plants reused, plus
new quieter ones — it raised recall by 2–9 points (2 on the new plants) at nearly the same
precision. A second plant is what would settle it.

## 4. The weekly plan

One row per train: an action, a week, the evidence, a confidence and an owner. The rules run in a
fixed order — manufacturer limit, too little data, at the level, near it or crossing within four
weeks, rising further out, flat or falling — and two trains are never planned into one week if that
leaves fewer than two running. Here is the plan as it would have read on day 700 of the record.

In [6]:
from chemai.evaluation.cip_plan import TrainRecord, build_plan

def plan_on(day):
    trains = []
    for c in cycles:
        if c.start <= day < c.end:
            m = c.abs_day <= day
            trains.append(TrainRecord(name=c.train, day=list(c.day[m]), dp_norm=list(c.dp_norm[m]),
                                      censored=list(c.censored[m])))
    return build_plan(trains, cfg, trains_total=3)

p = plan_on(700)
p.table()[["train", "category", "action", "week", "owner", "confidence", "evidence"]]

,train,category,action,week,owner,confidence,evidence
0,A,review,Decide this week: DP_norm 1.22 is at or above ...,1,process engineering,high,"level 1.225, trend +1.73 %/week"
1,B,review,Decide this week: DP_norm 1.18 is at or above ...,2,process engineering,high,"level 1.175, trend +2.31 %/week"
2,C,review,Decide this week: DP_norm 1.24 is at or above ...,3,process engineering,high,"level 1.235, trend +2.33 %/week"


## 5. The plan, replayed against what the operators did

Every 7 days over the whole record, what did the plan say in the week before each real cleaning?

In [7]:
rows = []
for day in range(14, int(daily["day_index"].max()) + 1, 7):
    for r in plan_on(day).rows:
        rows.append(dict(day=day, train=r.train, category=r.category, level_now=r.level_now))
replay = pd.DataFrame(rows)
before = []
for _, c in ledger.iterrows():
    w = replay[(replay["train"] == c["train"]) & (replay["day"] < c["day_index"]) & (replay["day"] >= c["day_index"] - 7)]
    before.append(dict(cleaning=c["episode"], day=c["day_index"], plan_week_before=w["category"].iloc[-1] if len(w) else "-",
                       fitted_level=w["level_now"].iloc[-1] if len(w) else None))
before = pd.DataFrame(before)
print(before["plan_week_before"].value_counts().to_dict())
print("share of all weekly rows that said 'data' (too few recorded days):", (replay["category"] == "data").mean().round(2))
before

{'review': 12, 'wait': 2, 'none': 2, 'plan': 1, 'watch': 1}
share of all weekly rows that said 'data' (too few recorded days): 0.32


,cleaning,day,plan_week_before,fitted_level
0,A#1,579,review,1.2219
1,A#2,727,review,1.2339
2,A#3,765,review,1.2010
3,B#1,456,review,1.3973
4,B#2,482,review,1.4024
5,B#3,524,review,1.3903
6,B#4,544,wait,NaN
7,B#5,584,none,1.0959
8,B#6,629,none,1.0397
9,B#7,726,review,1.1638


13 of the 18 cleanings came when the plan said *review* or *plan*. Two were separate campaigns 18
and 21 days after the previous one (*wait*). Three — B#5, B#6 and C#1 — came with no hydraulic
reason: the last week's median DP_norm was only 1.04–1.11. The deposit does not say why. Before
C#1, salt passage had been about 60 % above its baseline for a month: a reason a pressure-drop plan
cannot see.

## 6. The live service

The same plan is served at `https://chemai-tr2c.onrender.com/ro/` (free hosting: the first request
after a quiet spell can take a minute):

```python
import requests
r = requests.post("https://chemai-tr2c.onrender.com/ro/plan", json={
    "trains": [{"name": "A", "day": list(range(43)), "dp_norm": [1.05 + 0.0016 * d for d in range(43)]}],
    "review_level": 1.15})
r.json()["rows"]
```

## What this notebook settled

- On the one real plant, **the date forecast did not beat a simple margin rule**, and its level
  forecasts were **no better than persistence**.
- It was **blind while the pressure drop sat on its baseline**, and its long-range dates were late —
  and **flagged**.
- The weekly plan uses **both** triggers, a change made after this test and labelled as such.
- **One plant is not a validation.** What would settle it is a second plant with the raw flows,
  pressures and temperatures, so the normalisation can be done with the plant's own exponent.